In [9]:
import pandas as pd
import numpy as np
import scipy.stats as sm
import matplotlib.pyplot as plt
import yfinance as yf
 #Vanguard S&P 500 ETF
 #Vanguard Total Stock Market ETF
 #Vanguard Morningstar Large Cap ETF
 #Vanguard Minimum Volatility ETF
 #Vanguard Information Technology ETF

etfs = ['VOO','VTI','VV','VFMV','VGT']
data = []
for i in etfs:
    series = yf.download(i, start='2010-01-01', end='2026-09-23')['Close']
    if isinstance(series.index, pd.MultiIndex):
        series = series.get_level_values(0)
    data.append(series)


etf_data = pd.concat(data, axis = 1)
etf_data = etf_data.dropna()


#Get returns
returns = etf_data.pct_change().dropna()
monthly_returns = returns.resample('M').apply(lambda x: (1+x).prod() - 1).dropna()
monthly_returns



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
C:\Users\rnana\AppData\Local\Temp\ipykernel_53220\4280813284.py:27: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_returns = returns.resample('M').apply(lambda x: (1+x).prod() - 1).dropna()


Ticker,VOO,VTI,VV,VFMV,VGT
Date,,,,,
2018-02-28,-0.006495,-0.007497,-0.006055,-0.006540,0.014589
2018-03-31,-0.024676,-0.019632,-0.024481,0.022775,-0.033753
2018-04-30,0.003470,0.004494,0.003135,0.015704,-0.000234
2018-05-31,0.024164,0.027213,0.024426,0.018616,0.070484
2018-06-30,0.007585,0.007035,0.006089,0.020419,-0.006099
...,...,...,...,...,...
2026-05-31,0.052847,0.051838,0.054124,0.004436,0.173289
2026-06-30,-0.009605,-0.003853,-0.009555,-0.009046,-0.011546
2026-07-31,-0.000233,-0.004945,-0.000901,0.032677,-0.053296


In [24]:
#6 years of backtesting due to minimum volatility etf starting in 2018. To avoid look ahead bias, we will start in 2021, rebalancing monthly and using a rollback of 36 months to estimate mu and covariance.
'''
For example:

    2021-01-31 use previous 36 months(all the way back to 2018) to calculate optimal weights using mu and covariance then test it on 2021 realized returns. Then move to 2021-02-28 and use previous 36 months to calculate optimal weights(rebalancing)
'''

lookback = 36
portfolio_returns = []
weight_history = []
dates = []
cash_weight = 0.10
from scipy.optimize import minimize
for i in range(lookback, len(monthly_returns)):
    #Train the first 36 months of data 
    train = monthly_returns.iloc[i-lookback:i]
    #Calculate the mean and covariance of training data
    mu = train.mean()
    covariance = train.cov()
    

    def negative_sharpe(weights, mu, covariance, risk_free_rate=0.0):  #Negative sharpe because there is no maximize function so minimizing a negative sharpe ratio is equivalent
         cash_weight = 0.10
         portfolio_return = np.dot(weights, mu) + cash_weight * risk_free_rate
         portfolio_volatility = np.sqrt(np.dot(weights.T, np.dot(covariance, weights)))
         sharpe_ratio = (portfolio_return - risk_free_rate) / portfolio_volatility
         return -sharpe_ratio

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 0.90}) #Sum of weights have to equal 1
    bounds = [(0,1)] * len(mu) #Must be between 0 and 1
    x0 = np.ones(len(mu))/len(mu) #Initial guesses
    results = minimize(negative_sharpe,x0, method='SLSQP', bounds=bounds, constraints=constraints, args=(mu,covariance))
    weights = results.x

    next_month_return = monthly_returns.iloc[i]
    portfolio_return = np.dot(weights, next_month_return) + cash_weight * next_month_return
    portfolio_returns.append(portfolio_return)
    weight_history.append(weights)
    dates.append(monthly_returns.index[i])

portfolio_weighted_returns = pd.DataFrame(portfolio_returns, index=dates, columns=['Portfolio Returns'])
weights = pd.DataFrame(weight_history, index=dates, columns=etfs)
weights



#portfolio_weighted_returns #Out of sample rolling backtest


#Potential problem: This is starting in 2021 and only using a lookback of 36 months aka 3 years which may not be sufficient enough in determining especially if the returns are noisy

#cumulative_returns = (1+portfolio_weighted_returns).cumprod() 
#cumulative_returns

    
         


,VOO,VTI,VV,VFMV,VGT
2021-02-28,1.804112e-16,6.938894e-17,0.000000e+00,2.151057e-16,0.900000
2021-03-31,1.551584e-18,1.301775e-16,9.540979e-17,0.000000e+00,0.900000
2021-04-30,0.000000e+00,0.000000e+00,0.000000e+00,1.387779e-17,0.900000
2021-05-31,0.000000e+00,0.000000e+00,0.000000e+00,5.811324e-17,0.900000
2021-06-30,1.179204e-16,6.358971e-17,3.122502e-17,0.000000e+00,0.900000
...,...,...,...,...,...
2026-05-31,0.000000e+00,0.000000e+00,0.000000e+00,6.106233e-01,0.289377
2026-06-30,4.540986e-02,0.000000e+00,1.859407e-17,6.243231e-01,0.230267
2026-07-31,0.000000e+00,6.681158e-17,6.674311e-18,6.398319e-01,0.260168
2026-08-31,6.028164e-17,0.000000e+00,8.890458e-18,7.008785e-01,0.199122


In [27]:
from scipy.optimize import minimize

cash_weight = 0.10

# Use all historical data from 2018 through the latest completed month
train = monthly_returns.loc['2018-01-01':'2026-08-31']

# Calculate mean returns and covariance
mu = train.mean()
covariance = train.cov()


def negative_sharpe(weights, mu, covariance, risk_free_rate=0.0):

    portfolio_return = np.dot(weights, mu) + cash_weight * risk_free_rate

    portfolio_volatility = np.sqrt(
        np.dot(weights.T, np.dot(covariance, weights))
    )

    if portfolio_volatility <= 0:
        return 1e10

    sharpe_ratio = (
        portfolio_return - risk_free_rate
    ) / portfolio_volatility

    return -sharpe_ratio


# ETF weights must sum to 90%
constraints = {
    'type': 'eq',
    'fun': lambda x: np.sum(x) - 0.90
}

# Long-only
bounds = [(0, 0.90)] * len(mu)

# Equal-weight initial guess
x0 = np.ones(len(mu)) * (0.90 / len(mu))

# Optimize ONCE
results = minimize(
    negative_sharpe,
    x0,
    method='SLSQP',
    bounds=bounds,
    constraints=constraints,
    args=(mu, covariance),
    options={'maxiter': 1000, 'ftol': 1e-12}
)

if not results.success:
    raise RuntimeError(results.message)

weights = results.x


# Create allocation DataFrame
allocation = pd.DataFrame({
    'ETF': train.columns,
    'Weight': weights
})

# Add cash allocation
allocation.loc[len(allocation)] = ['Cash', cash_weight]

# Convert to percentage
allocation['Allocation (%)'] = allocation['Weight'] * 100

allocation.rank(ascending=False)
allocation

,ETF,Weight,Allocation (%)
0,VOO,4.166649e-17,4.166649e-15
1,VTI,9.270393e-18,9.270393e-16
2,VV,0.000000e+00,0.000000e+00
3,VFMV,2.129636e-01,2.129636e+01
4,VGT,6.870364e-01,6.870364e+01
5,Cash,1.000000e-01,1.000000e+01


In [28]:
train.corr()

Ticker,VOO,VTI,VV,VFMV,VGT
Ticker,,,,,
VOO,1.000000,0.996317,0.999152,0.876787,0.906850
VTI,0.996317,1.000000,0.997212,0.880929,0.901664
VV,0.999152,0.997212,1.000000,0.871189,0.913200
VFMV,0.876787,0.880929,0.871189,1.000000,0.674319
VGT,0.906850,0.901664,0.913200,0.674319,1.000000


In [ ]:
#Model B Exponentially Weighted Average Optimization Model
'''
EWMA: Weights recent dates higher than older dates. The formula for weights is wk = (1-lambda)*lambda**k where k is the most recent month. k = 0, most recent month
New strategies to test: SMA Crossover, Bollinger Bands


'''


def ewma_weights(n, lambda_):
    